In [1]:
import sys
!{sys.executable} -m pip install --upgrade --force-reinstall \
    azureml-widgets \
    azureml-sdk[automl,notebooks,train] \
    azureml-train-automl-runtime \
    pandas \
    scikit-learn


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
INFO: pip is looking at multiple versions of ipykernel to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 18.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.2/26.2 MB 67.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 92.7 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 135.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 92.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 49.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.2 MB/s  0:00:00
   ━━━

In [2]:
import azureml.core
from azureml.core import Workspace, Experiment, Environment, ScriptRunConfig
from azureml.widgets import RunDetails

# These are the absolute direct paths
import azureml.train.hyperdrive as hd
from azureml.train.hyperdrive.runconfig import HyperDriveConfig
from azureml.train.hyperdrive.sampling import RandomParameterSampling
from azureml.train.hyperdrive.policy import BanditPolicy
from azureml.train.hyperdrive.run import PrimaryMetricGoal
from azureml.train.hyperdrive.parameter_expressions import choice, uniform

print(f"✅ Success! Using SDK Version: {azureml.core.VERSION}")


✅ Success! Using SDK Version: 1.61.0


In [3]:
from azureml.core import Workspace, Experiment

ws = Workspace.from_config()
exp = Experiment(workspace=ws, name="udacity-project")

print('Workspace name: ' + ws.name, 
      'Azure region: ' + ws.location, 
      'Subscription id: ' + ws.subscription_id, 
      'Resource group: ' + ws.resource_group, sep = '\n')

run = exp.start_logging()


Workspace name: quick-starts-ws-298579
Azure region: southcentralus
Subscription id: 1b944a9b-fdae-4f97-aeb1-b7eea0beac53
Resource group: aml-quickstarts-298579


In [4]:
from azureml.core.compute import ComputeTarget, AmlCompute
from azureml.core.compute_target import ComputeTargetException

cluster_name = "cpu-cluster"

try:
    cpu_cluster = ComputeTarget(workspace=ws, name=cluster_name)
    print('Found existing cluster, use it.')
except ComputeTargetException:
    compute_config = AmlCompute.provisioning_configuration(vm_size='Standard_D2_V2',
                                                           max_nodes=4)
    cpu_cluster = ComputeTarget.create(ws, cluster_name, compute_config)

cpu_cluster.wait_for_completion(show_output=True)


InProgress..
SucceededProvisioning operation finished, operation "Succeeded"
Succeeded
AmlCompute wait for completion finished

Minimum number of nodes requested have been provisioned


In [5]:
import azureml.core
from azureml.core import Workspace, Experiment, Environment, ScriptRunConfig
from azureml.widgets import RunDetails
# Absolute imports to bypass the 'azureml.train' module error
from azureml.train.hyperdrive.runconfig import HyperDriveConfig
from azureml.train.hyperdrive.run import PrimaryMetricGoal
from azureml.train.hyperdrive.policy import BanditPolicy
from azureml.train.hyperdrive.sampling import RandomParameterSampling
from azureml.train.hyperdrive.parameter_expressions import choice, uniform


# 1. Parameter sampling
ps = RandomParameterSampling({
    "C": uniform(0.01, 1.0),
    "max_iter": choice(100, 200, 300),
    "penalty": choice("l1", "l2"),
    "solver": choice("liblinear", "saga")
})

# 2. Early termination policy
policy = BanditPolicy(evaluation_interval=2, slack_factor=0.1)

# 3. Environment and Config
sklearn_env = Environment.from_conda_specification(name='sklearn-env', file_path='conda_dependencies.yml')

src = ScriptRunConfig(source_directory='.',
                      script='train.py',
                      compute_target=cpu_cluster,
                      environment=sklearn_env)

# 4. HyperDrive Configuration
hyperdrive_config = HyperDriveConfig(
                    run_config=src,
                    hyperparameter_sampling=ps,
                    policy=policy,
                    primary_metric_name='AUC', 
                    primary_metric_goal=PrimaryMetricGoal.MAXIMIZE,
                    max_total_runs=20,
                    max_concurrent_runs=4
)

In [9]:
# Submit your hyperdrive run to the experiment and show run details with the widget.

hyperdrive_run = exp.submit(hyperdrive_config)

# 6. VISUALIZE
hyperdrive_run.wait_for_completion(show_output=True)

RunId: HD_611975fc-fa23-42b0-bffd-6612c5acc9ca
Web View: https://ml.azure.com/runs/HD_611975fc-fa23-42b0-bffd-6612c5acc9ca?wsid=/subscriptions/1b944a9b-fdae-4f97-aeb1-b7eea0beac53/resourcegroups/aml-quickstarts-298579/workspaces/quick-starts-ws-298579&tid=660b3398-b80e-49d2-bc5b-ac1dc93b5254

Streaming azureml-logs/hyperdrive.txt

[2026-04-21T19:01:25.1052431Z][GENERATOR][DEBUG]Sampled 4 jobs from search space 
[2026-04-21T19:01:25.3281108Z][SCHEDULER][INFO]Scheduling job, id='HD_611975fc-fa23-42b0-bffd-6612c5acc9ca_0' 
[2026-04-21T19:01:25.4562995Z][SCHEDULER][INFO]Scheduling job, id='HD_611975fc-fa23-42b0-bffd-6612c5acc9ca_1' 
[2026-04-21T19:01:25.4540046Z][SCHEDULER][INFO]Scheduling job, id='HD_611975fc-fa23-42b0-bffd-6612c5acc9ca_3' 
[2026-04-21T19:01:25.4552488Z][SCHEDULER][INFO]Scheduling job, id='HD_611975fc-fa23-42b0-bffd-6612c5acc9ca_2' 
[2026-04-21T19:01:25.6836611Z][SCHEDULER][INFO]Successfully scheduled a job. Id='HD_611975fc-fa23-42b0-bffd-6612c5acc9ca_0' 
[2026-04-21T19:0

{'runId': 'HD_611975fc-fa23-42b0-bffd-6612c5acc9ca',
 'target': 'cpu-cluster',
 'status': 'Completed',
 'startTimeUtc': '2026-04-21T19:01:23.027766Z',
 'endTimeUtc': '2026-04-21T19:11:00.75241Z',
 'services': {},
 'properties': {'primary_metric_config': '{"name":"AUC","goal":"maximize"}',
  'resume_from': 'null',
  'runTemplate': 'HyperDrive',
  'azureml.runsource': 'hyperdrive',
  'platform': 'AML',
  'ContentSnapshotId': 'f374507f-a9de-41ca-a781-9a8149738783',
  'user_agent': 'python/3.10.11 (Linux-6.8.0-1044-azure-x86_64-with-glibc2.35) msrest/0.7.1 Hyperdrive.Service/1.0.0 Hyperdrive.SDK/core.1.61.0',
  'best_child_run_id': 'HD_611975fc-fa23-42b0-bffd-6612c5acc9ca_7',
  'score': '0.9226081649059826',
  'best_metric_status': 'Succeeded',
  'best_data_container_id': 'dcid.HD_611975fc-fa23-42b0-bffd-6612c5acc9ca_7'},
 'inputDatasets': [],
 'outputDatasets': [],
 'runDefinition': {'configuration': None,
  'attribution': None,
  'telemetryValues': {'amlClientType': 'azureml-sdk-train',


In [10]:
%pip install joblib

Note: you may need to restart the kernel to use updated packages.


In [11]:
import joblib
# Get your best run and save the model from that run.
# 1. Submit the experiment and wait for it to finish completely

hyperdrive_run = exp.submit(hyperdrive_config)
hyperdrive_run.wait_for_completion(show_output=True)

best_run = hyperdrive_run.get_best_run_by_primary_metric()

if best_run:
    best_run_metrics = best_run.get_metrics()
    
    print('✅ Best AUC:', best_run_metrics.get('AUC'))
    print('Accuracy:', best_run_metrics.get('Accuracy'))

    model = best_run.register_model(
        model_name='hyperdrive_best_model', 
        model_path='outputs/model.joblib'
    )
    

RunId: HD_f7e125a5-e62b-4281-9d5a-95f5722ed896
Web View: https://ml.azure.com/runs/HD_f7e125a5-e62b-4281-9d5a-95f5722ed896?wsid=/subscriptions/1b944a9b-fdae-4f97-aeb1-b7eea0beac53/resourcegroups/aml-quickstarts-298579/workspaces/quick-starts-ws-298579&tid=660b3398-b80e-49d2-bc5b-ac1dc93b5254

Streaming azureml-logs/hyperdrive.txt

[2026-04-21T19:18:24.8626205Z][GENERATOR][DEBUG]Sampled 4 jobs from search space 
[2026-04-21T19:18:25.2259573Z][SCHEDULER][INFO]Scheduling job, id='HD_f7e125a5-e62b-4281-9d5a-95f5722ed896_0' 
[2026-04-21T19:18:25.3233324Z][SCHEDULER][INFO]Scheduling job, id='HD_f7e125a5-e62b-4281-9d5a-95f5722ed896_1' 
[2026-04-21T19:18:25.3242975Z][SCHEDULER][INFO]Scheduling job, id='HD_f7e125a5-e62b-4281-9d5a-95f5722ed896_2' 
[2026-04-21T19:18:25.4596996Z][SCHEDULER][INFO]Scheduling job, id='HD_f7e125a5-e62b-4281-9d5a-95f5722ed896_3' 
[2026-04-21T19:18:25.7425632Z][SCHEDULER][INFO]Successfully scheduled a job. Id='HD_f7e125a5-e62b-4281-9d5a-95f5722ed896_1' 
[2026-04-21T19:1

In [15]:
from azureml.data.dataset_factory import TabularDatasetFactory

# Create TabularDataset using TabularDatasetFactory
# Data is available at: 
# "https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv"
# Define the URL for the dataset

path = "https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv"
ds = TabularDatasetFactory.from_delimited_files(path=path)

print("Dataset loaded successfully!")


Dataset loaded successfully!


In [16]:
from train import clean_data
import pandas as pd
# Use the clean_data function to clean your data.
x, y = clean_data(ds)

df = pd.concat([x, y], axis=1)

print("Data cleaned and prepared.")


{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}
Data cleaned and prepared.


In [10]:
# =========================
# RESOURCE CLEANUP
# =========================

cpu_cluster.delete()
print("Compute cluster deleted!")


Compute cluster deleted!
